# Synthetic Data Generator

Generate any kind of dataset with right descriptive prompt.
This uses Hugging face model LLAMA 3.1-8B which can be changed for any other model by just changing the variable HF_MODEL value.

Input and Output are encoded and decoded using tokenizer.
Quantization is used to create model object.
The generated data output is shown after slicing off the previous input prompt content.

Interactive interface is made using Gradio UI.

### Google Colab link:
https://colab.research.google.com/drive/1h3iaM6HeMAPR_V0KYQ5uyyp_WwwwEOwW?usp=sharing

Run this file on T4 GPU instance.


In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
import gradio as gr
import os
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

In [ ]:
load_dotenv()

hf_token = os.getenv('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)

In [ ]:
HF_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Tokenizer for model
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)
tokenizer.pad_token = tokenizer.eos_token

# Quantization for model
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(HF_MODEL, device_map="auto", quantization_config=quant_config)

In [ ]:
system_prompt = """Your are a synthetic data generator. Your task is to output the generated data in structured format.
    User input given will be:
        - Description: The type of dataset requested by the user. Analyse it and extract any data features provided here.
        - Rows: Number of fake records to be generated.
        - Columns: Number of features that should be generated in each record. Use all the features provided by user and any relevant features that can help the dataset more to form the schema.

    Instructions:
        - No extra expalnations, no markdowns and no extra text.
        - Generate realistic but fake data. Do not use real names, emails or any sensitive information that can harm privacy.
        - Use consistent writing style, schema format and conventions.

    Output format:

"""

csv_prompt = """
    - Output should be in CSV format.
    - First row is header row, and then one record per row.
    - Use commas; escape quotes inside fields.
    """

json_prompt = """
    - Output should be in JSON array of objects.
    - The keys should be consistent in each object.
    - Each object is one record
    """

In [ ]:
def generate_prompts(description, rows, cols, output_format):
    system_msg = system_prompt
    if output_format == "CSV":
        system_msg += csv_prompt
    else:
        system_msg += json_prompt

    user_msg = f"""
        Generate synthetic data with the following information.
            - Description: {description}
            - Rows: {rows}
            - Columns: {cols}

        The output should be in {output_format} format only. Do not add any extra text, markdown or information.
    """

    messages = [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}]
    return messages

In [ ]:
def generate_dataset(description: str, rows: int, cols: int, output_format: str):
    description = description.strip()
    if not description:
        return "Please give a dataset description to generate dataset as per requirement.(E.g. wheat production in last 20 years as per rainfall, region, manure, soil)"

    messages = generate_prompts(description, rows, cols, output_format)

    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

    outputs = model.generate(input_ids=inputs, max_new_tokens=2000, temperature=0.7, do_sample=True)

    return tokenizer.decode(outputs[0][inputs.shape[1]+4:], skip_special_tokens=True)


In [ ]:
with gr.Blocks() as interface:
    gr.Markdown("""
        ## Customizable Synthetic data Generator
        Describe the dataset you want and any fields or features you might want.
        Choose number of rows (data records) and columns (data features).
        Choose output format (CSV or JSON)""")

    with gr.Row():
        description = gr.Textbox(
            label="Dataset description",
            placeholder="E.g. wheat production in last 20 years as per rainfall, region, manure, soil",
            lines=4
        )

    with gr.Row():
        rows = gr.Slider(1,50, value=5, step=1, label="Number of rows (data records)")
        cols = gr.Slider(1,15, value=5, step=1, label="Number of columns (data features)")

    output_format = gr.Dropdown(choices=["CSV", "JSON"], value="CSV", label="Output Format")

    button = gr.Button("Generate Dataset", variant="primary")

    output = gr.Textbox(
            label="Generated Dataset",
            placeholder="The generated synthetic dataset will be displayed here",
            lines=10
        )

    button.click(fn=generate_dataset, inputs=[description, rows, cols, output_format], outputs=output)

interface.launch()
